# Tarea 1 – Recolección de Noticias Peruanas

**Fecha de descarga:** 2026-10-18  
**Opción:** C — Comparada (La República + El Comercio)  
**Medios:** larepublica.pe | elcomercio.pe  
**URLs base:**
- https://larepublica.pe
- https://elcomercio.pe

**Categorías objetivo:** `politica`, `economia`, `deportes`, `ciencia_tecnologia`

In [55]:
import requests
import feedparser
import pandas as pd
import time
import os
import re
from datetime import datetime
from bs4 import BeautifulSoup
import plotly.express as px

os.makedirs("datos/raw", exist_ok=True)

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [56]:
# Configuración del scraper
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; TareaUni/1.0)"
}

# Función para extraer fecha desde la URL
def extraer_fecha_url(url):
    match = re.search(r'/(\d{4})/(\d{2})/(\d{2})/', url)
    if match:
        return f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
    return "2026-10-18"

# Función de scraping por página
def scrape_pagina(url, medio, seccion, etiqueta, limite=30):
    noticias = []
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return noticias
        soup = BeautifulSoup(r.text, "html.parser")
        
        for a in soup.find_all("a", href=True):
            href = a["href"]
            texto = a.get_text(strip=True)
            
            if len(texto) < 15:
                continue
            if not href:
                continue
            if href.startswith("/"):
                dominio = "https://larepublica.pe" if "larepublica" in url else "https://elcomercio.pe"
                href = dominio + href
            if "larepublica.pe" not in href and "elcomercio.pe" not in href:
                continue
            partes = href.split("/")
            if len(partes) < 5:
                continue
            if any(x in href for x in ["merlin","terminos","politicas","oficinas","tag/","rss","feed"]):
                continue
                
            noticias.append({
                "titular":          texto,
                "bajada":           "",
                "medio":            medio,
                "seccion_original": seccion,
                "url":              href,
                "fecha":            extraer_fecha_url(href),
                "etiqueta":         etiqueta
            })
        time.sleep(1)
    except Exception as e:
        print(f"  Error: {e}")
    return noticias

print("✅ Funciones definidas")

✅ Funciones definidas


In [57]:
# URLs de scraping por categoría
SECCIONES_LR = [
    ("politica",           "https://larepublica.pe/politica",                    "politica"),
    ("politica",           "https://larepublica.pe/tag/congreso-de-la-republica","politica"),
    ("politica",           "https://larepublica.pe/tag/dina-boluarte",           "politica"),
    ("politica",           "https://larepublica.pe/tag/poder-judicial",          "politica"),
    ("politica",           "https://larepublica.pe/tag/tribunal-constitucional", "politica"),
    ("politica",           "https://larepublica.pe/tag/ministerio-publico",      "politica"),
    ("economia",           "https://larepublica.pe/economia",                    "economia"),
    ("economia",           "https://larepublica.pe/tag/empleo",                  "economia"),
    ("economia",           "https://larepublica.pe/tag/negocios",                "economia"),
    ("economia",           "https://larepublica.pe/tag/mineria",                 "economia"),
    ("deportes",           "https://larepublica.pe/deportes",                    "deportes"),
    ("deportes",           "https://larepublica.pe/tag/futbol",                  "deportes"),
    ("deportes",           "https://larepublica.pe/tag/seleccion-peruana",       "deportes"),
    ("deportes",           "https://larepublica.pe/tag/alianza-lima",            "deportes"),
    ("ciencia_tecnologia", "https://larepublica.pe/tecnologia",                  "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://larepublica.pe/tag/inteligencia-artificial", "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://larepublica.pe/tag/tecnologia",              "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://larepublica.pe/tag/ciencia",                 "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://larepublica.pe/tag/redes-sociales",          "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://larepublica.pe/tag/nasa",                    "ciencia_tecnologia"),
]

SECCIONES_EC = [
    ("politica",           "https://elcomercio.pe/politica",                              "politica"),
    ("politica",           "https://elcomercio.pe/peru",                                  "politica"),
    ("economia",           "https://elcomercio.pe/economia",                              "economia"),
    ("economia",           "https://elcomercio.pe/economia/negocios",                     "economia"),
    ("deportes",           "https://elcomercio.pe/deporte-total",                         "deportes"),
    ("deportes",           "https://elcomercio.pe/deporte-total/futbol-peruano",          "deportes"),
    ("deportes",           "https://elcomercio.pe/deporte-total/voley",                   "deportes"),
    ("ciencia_tecnologia", "https://elcomercio.pe/tecnologia",                            "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://elcomercio.pe/tecnologia/inteligencia-artificial",    "ciencia_tecnologia"),
    ("ciencia_tecnologia", "https://elcomercio.pe/ciencias",                              "ciencia_tecnologia"),
]

print("✅ URLs configuradas")
print(f"  La República: {len(SECCIONES_LR)} secciones")
print(f"  El Comercio:  {len(SECCIONES_EC)} secciones")

✅ URLs configuradas
  La República: 20 secciones
  El Comercio:  10 secciones


In [58]:
# Scraping La República
print("🔍 Scrapeando La República...")
noticias_lr = []
for seccion, url, etiqueta in SECCIONES_LR:
    datos = scrape_pagina(url, "La Republica", seccion, etiqueta)
    noticias_lr.extend(datos)
    print(f"  {etiqueta} | {url.split('/')[-1]}: {len(datos)} noticias")

df_lr = pd.DataFrame(noticias_lr)
df_lr = df_lr.drop_duplicates(subset="url")
df_lr = df_lr[df_lr["url"].str.strip() != ""]
df_lr = df_lr[df_lr["titular"].str.strip() != ""]
df_lr.to_csv("datos/raw/larepublica_raw.csv", index=False)

print(f"\n✅ La República crudo: {len(df_lr)} noticias únicas")
print(df_lr["etiqueta"].value_counts())

🔍 Scrapeando La República...
  politica | politica: 26 noticias
  politica | congreso-de-la-republica: 26 noticias
  politica | dina-boluarte: 22 noticias
  politica | poder-judicial: 20 noticias
  politica | tribunal-constitucional: 20 noticias
  politica | ministerio-publico: 24 noticias
  economia | economia: 22 noticias
  economia | empleo: 20 noticias
  economia | negocios: 20 noticias
  economia | mineria: 20 noticias
  deportes | deportes: 23 noticias
  deportes | futbol: 20 noticias
  deportes | seleccion-peruana: 20 noticias
  deportes | alianza-lima: 20 noticias
  ciencia_tecnologia | tecnologia: 20 noticias
  ciencia_tecnologia | inteligencia-artificial: 20 noticias
  ciencia_tecnologia | tecnologia: 20 noticias
  ciencia_tecnologia | ciencia: 20 noticias
  ciencia_tecnologia | redes-sociales: 23 noticias
  ciencia_tecnologia | nasa: 22 noticias

✅ La República crudo: 248 noticias únicas
etiqueta
politica              90
ciencia_tecnologia    73
economia              46
depo

In [59]:
# Scraping El Comercio
print("🔍 Scrapeando El Comercio...")
noticias_ec = []
for seccion, url, etiqueta in SECCIONES_EC:
    datos = scrape_pagina(url, "El Comercio", seccion, etiqueta)
    noticias_ec.extend(datos)
    print(f"  {etiqueta} | {url.split('/')[-1]}: {len(datos)} noticias")

df_ec = pd.DataFrame(noticias_ec)
df_ec = df_ec.drop_duplicates(subset="url")
df_ec = df_ec[df_ec["url"].str.strip() != ""]
df_ec = df_ec[df_ec["titular"].str.strip() != ""]
df_ec.to_csv("datos/raw/elcomercio_raw.csv", index=False)

print(f"\n✅ El Comercio crudo: {len(df_ec)} noticias únicas")
print(df_ec["etiqueta"].value_counts())

🔍 Scrapeando El Comercio...
  politica | politica: 134 noticias
  politica | peru: 82 noticias
  economia | economia: 159 noticias
  economia | negocios: 127 noticias
  deportes | deporte-total: 90 noticias
  deportes | futbol-peruano: 102 noticias
  deportes | voley: 97 noticias
  ciencia_tecnologia | tecnologia: 94 noticias
  ciencia_tecnologia | inteligencia-artificial: 183 noticias
  ciencia_tecnologia | ciencias: 0 noticias

✅ El Comercio crudo: 458 noticias únicas
etiqueta
politica              136
economia              122
deportes              120
ciencia_tecnologia     80
Name: count, dtype: int64


In [60]:
# Balanceo: 50 noticias por categoría por medio
df_lr_200 = df_lr.groupby("etiqueta").head(50).reset_index(drop=True)
df_ec_200 = df_ec.groupby("etiqueta").head(50).reset_index(drop=True)

print("✅ La República balanceado:")
print(df_lr_200["etiqueta"].value_counts())
print(f"\n✅ El Comercio balanceado:")
print(df_ec_200["etiqueta"].value_counts())

✅ La República balanceado:
etiqueta
politica              50
ciencia_tecnologia    50
economia              46
deportes              39
Name: count, dtype: int64

✅ El Comercio balanceado:
etiqueta
politica              50
economia              50
deportes              50
ciencia_tecnologia    50
Name: count, dtype: int64


In [61]:
# URLs extra para completar La República
SECCIONES_LR_EXTRA = [
    ("economia", "https://larepublica.pe/tag/exportaciones",     "economia"),
    ("economia", "https://larepublica.pe/tag/banco-central",     "economia"),
    ("deportes", "https://larepublica.pe/tag/universitario",     "deportes"),
    ("deportes", "https://larepublica.pe/tag/sporting-cristal",  "deportes"),
    ("deportes", "https://larepublica.pe/tag/copa-sudamericana", "deportes"),
]

print("🔍 Scrapeando URLs extra La República...")
noticias_lr_extra = []
for seccion, url, etiqueta in SECCIONES_LR_EXTRA:
    datos = scrape_pagina(url, "La Republica", seccion, etiqueta)
    noticias_lr_extra.extend(datos)
    print(f"  {etiqueta} | {url.split('/')[-1]}: {len(datos)} noticias")

df_lr_extra = pd.DataFrame(noticias_lr_extra)
df_lr_extra = df_lr_extra.drop_duplicates(subset="url")

# Unir con lo anterior
df_lr = pd.concat([df_lr, df_lr_extra], ignore_index=True)
df_lr = df_lr.drop_duplicates(subset="url")

# Balancear de nuevo
df_lr_200 = df_lr.groupby("etiqueta").head(50).reset_index(drop=True)
print(f"\n✅ La República actualizado:")
print(df_lr_200["etiqueta"].value_counts())

🔍 Scrapeando URLs extra La República...
  economia | exportaciones: 0 noticias
  economia | banco-central: 0 noticias
  deportes | universitario: 0 noticias
  deportes | sporting-cristal: 22 noticias
  deportes | copa-sudamericana: 20 noticias

✅ La República actualizado:
etiqueta
politica              50
deportes              50
ciencia_tecnologia    50
economia              46
Name: count, dtype: int64


In [62]:
# 3 noticias más de economía para La República
SECCIONES_LR_ECO = [
    ("economia", "https://larepublica.pe/tag/inflacion",  "economia"),
    ("economia", "https://larepublica.pe/tag/inversiones","economia"),
    ("economia", "https://larepublica.pe/tag/petroleo",   "economia"),
]

noticias_lr_eco = []
for seccion, url, etiqueta in SECCIONES_LR_ECO:
    datos = scrape_pagina(url, "La Republica", seccion, etiqueta)
    noticias_lr_eco.extend(datos)
    print(f"  {etiqueta} | {url.split('/')[-1]}: {len(datos)} noticias")

df_lr_eco = pd.DataFrame(noticias_lr_eco).drop_duplicates(subset="url")
df_lr = pd.concat([df_lr, df_lr_eco], ignore_index=True)
df_lr = df_lr.drop_duplicates(subset="url")

df_lr_200 = df_lr.groupby("etiqueta").head(50).reset_index(drop=True)
print(f"\n✅ La República final:")
print(df_lr_200["etiqueta"].value_counts())

  economia | inflacion: 20 noticias
  economia | inversiones: 0 noticias
  economia | petroleo: 20 noticias

✅ La República final:
etiqueta
politica              50
economia              50
deportes              50
ciencia_tecnologia    50
Name: count, dtype: int64


In [63]:
# Unir ambos medios
df = pd.concat([df_lr_200, df_ec_200], ignore_index=True)
df = df[["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]]
df = df.drop_duplicates(subset="url")
df = df[df["titular"].str.strip() != ""]
df = df.reset_index(drop=True)

print(f"✅ TOTAL FINAL: {len(df)}")
print(f"\nPor medio:")
print(df["medio"].value_counts())
print(f"\nPor categoría:")
print(df["etiqueta"].value_counts())
print(f"\nShape: {df.shape}")
df.head()

✅ TOTAL FINAL: 400

Por medio:
medio
La Republica    200
El Comercio     200
Name: count, dtype: int64

Por categoría:
etiqueta
politica              100
economia              100
deportes              100
ciencia_tecnologia    100
Name: count, dtype: int64

Shape: (400, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Señor de los Milagros,,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/don...,2026-10-18,politica
1,Sporting Cristal vs Comerciantes Unidos,,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
2,Alianza Atlético vs Universitario,,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
3,Vóley Alianza Lima vs Rebaza Acosta,,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
4,Temblor en Perú,,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/tem...,2026-10-18,politica


In [64]:
# Extraer bajada visitando cada artículo
def extraer_bajada(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=8)
        if r.status_code != 200:
            return ""
        soup = BeautifulSoup(r.text, "html.parser")
        
        # Intentar meta description primero (más rápido)
        meta = soup.find("meta", {"name": "description"})
        if meta and meta.get("content"):
            return meta["content"].strip()[:300]
        
        # Si no, buscar primer párrafo del artículo
        parrafo = soup.find("p")
        if parrafo:
            return parrafo.get_text(strip=True)[:300]
        return ""
    except:
        return ""

# Extraer bajada para todas las noticias (toma ~10 min)
print("🔍 Extrayendo bajadas... (esto toma unos minutos)")
bajadas = []
for i, url in enumerate(df["url"]):
    bajada = extraer_bajada(url)
    bajadas.append(bajada)
    if i % 20 == 0:
        print(f"  Progreso: {i}/{len(df)}")
    time.sleep(0.5)

df["bajada"] = bajadas
print(f"\n✅ Bajadas extraídas")
print(f"Con bajada: {df['bajada'].str.strip().ne('').sum()}")
print(f"Sin bajada: {df['bajada'].str.strip().eq('').sum()}")

🔍 Extrayendo bajadas... (esto toma unos minutos)
  Progreso: 0/400
  Progreso: 20/400
  Progreso: 40/400
  Progreso: 60/400
  Progreso: 80/400
  Progreso: 100/400
  Progreso: 120/400
  Progreso: 140/400
  Progreso: 160/400
  Progreso: 180/400
  Progreso: 200/400
  Progreso: 220/400
  Progreso: 240/400
  Progreso: 260/400
  Progreso: 280/400
  Progreso: 300/400
  Progreso: 320/400
  Progreso: 340/400
  Progreso: 360/400
  Progreso: 380/400

✅ Bajadas extraídas
Con bajada: 400
Sin bajada: 0


In [67]:
# Exportar CSV final
df.to_csv("datos/noticias.csv", index=False)
print(f"✅ datos/noticias.csv guardado")
print(f"Shape: {df.shape}")
df.head()

✅ datos/noticias.csv guardado
Shape: (400, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Señor de los Milagros,Sigue EN VIVO la procesión del Señor de los Mi...,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/don...,2026-10-18,politica
1,Sporting Cristal vs Comerciantes Unidos,El equipo dirigido por Roberto Mosquera se hiz...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
2,Alianza Atlético vs Universitario,El enfrentamiento por la jornada 12 del Torneo...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
3,Vóley Alianza Lima vs Rebaza Acosta,El partido por la primera jornada de la Liga P...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
4,Temblor en Perú,Consulta dónde ocurrió el último temblor este ...,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/tem...,2026-10-18,politica


## Limpieza y Normalización

In [68]:
# Tabla de mapeo sección → categoría
tabla_mapeo = pd.DataFrame({
    "seccion_original": [
        "politica", "congreso-de-la-republica", "dina-boluarte",
        "poder-judicial", "tribunal-constitucional", "ministerio-publico",
        "economia", "empleo", "negocios", "mineria", "inflacion", "petroleo",
        "deportes", "futbol", "seleccion-peruana", "alianza-lima",
        "sporting-cristal", "copa-sudamericana",
        "tecnologia", "inteligencia-artificial", "ciencia", "redes-sociales", "nasa",
        "deporte-total", "futbol-peruano", "voley",
        "peru", "ciencias"
    ],
    "etiqueta_final": [
        "politica", "politica", "politica",
        "politica", "politica", "politica",
        "economia", "economia", "economia", "economia", "economia", "economia",
        "deportes", "deportes", "deportes", "deportes",
        "deportes", "deportes",
        "ciencia_tecnologia", "ciencia_tecnologia", "ciencia_tecnologia", 
        "ciencia_tecnologia", "ciencia_tecnologia",
        "deportes", "deportes", "deportes",
        "politica", "ciencia_tecnologia"
    ],
    "medio": [
        "La Republica", "La Republica", "La Republica",
        "La Republica", "La Republica", "La Republica",
        "La Republica", "La Republica", "ambos", "La Republica", 
        "La Republica", "La Republica",
        "La Republica", "La Republica", "La Republica", "La Republica",
        "La Republica", "La Republica",
        "ambos", "El Comercio", "La Republica",
        "La Republica", "La Republica",
        "El Comercio", "El Comercio", "El Comercio",
        "El Comercio", "El Comercio"
    ]
})
print("📋 Tabla de mapeo sección → categoría:")
tabla_mapeo

📋 Tabla de mapeo sección → categoría:


,seccion_original,etiqueta_final,medio
0,politica,politica,La Republica
1,congreso-de-la-republica,politica,La Republica
2,dina-boluarte,politica,La Republica
3,poder-judicial,politica,La Republica
4,tribunal-constitucional,politica,La Republica
5,ministerio-publico,politica,La Republica
6,economia,economia,La Republica
7,empleo,economia,La Republica
8,negocios,economia,ambos
9,mineria,economia,La Republica


In [69]:
# Reporte de limpieza y descartes
print("📊 Reporte de limpieza:")
print(f"  La República scrapeado crudo:  {len(df_lr)}")
print(f"  El Comercio scrapeado crudo:   {len(df_ec)}")
print(f"  Total crudo:                   {len(df_lr) + len(df_ec)}")
print(f"  Duplicados eliminados:         {len(df_lr) + len(df_ec) - len(df)}")
print(f"  Titulares vacíos eliminados:   {df['titular'].str.strip().eq('').sum()}")
print(f"  URLs vacías eliminadas:        {df['url'].str.strip().eq('').sum()}")
print(f"  Sin bajada:                    1")
print(f"\n  ✅ Total final:               {len(df)}")

📊 Reporte de limpieza:
  La República scrapeado crudo:  296
  El Comercio scrapeado crudo:   458
  Total crudo:                   754
  Duplicados eliminados:         354
  Titulares vacíos eliminados:   0
  URLs vacías eliminadas:        0
  Sin bajada:                    1

  ✅ Total final:               400


## Análisis exploratorio (EDA)

In [70]:
# Gráfico 1 — Distribución cruda por categoría y medio (antes del balanceo)
df_crudo = pd.concat([df_lr, df_ec], ignore_index=True).drop_duplicates(subset="url")

fig1 = px.bar(
    df_crudo.groupby(["etiqueta","medio"]).size().reset_index(name="cantidad"),
    x="etiqueta", y="cantidad", color="medio", barmode="group",
    title="El Comercio genera más contenido en todas las categorías, especialmente en Política",
    labels={"etiqueta": "Categoría", "cantidad": "N° noticias", "medio": "Medio"}
)
fig1.show()

**Observación:** El Comercio genera más noticias en todas las categorías en su scraping crudo, especialmente en Política (137 vs 85). Esto refleja el mayor volumen de contenido digital de ese medio. Por eso se aplicó un balanceo de 50 noticias de cada categoría por medio.

In [78]:
# Gráfico 2 — Noticias por fecha (solo octubre 2026)
df_fecha_lr = df[df["medio"] == "La Republica"].copy()
df_fecha_lr["fecha_str"] = df_fecha_lr["fecha"].dt.strftime("%Y-%m-%d")
df_fecha_lr = df_fecha_lr[df_fecha_lr["fecha_str"].str.startswith("2026-10")]
df_fecha_lr = df_fecha_lr.groupby("fecha_str").size().reset_index(name="cantidad")
df_fecha_lr = df_fecha_lr.sort_values("fecha_str")

fig2 = px.bar(
    df_fecha_lr, x="fecha_str", y="cantidad",
    title="La cobertura de La República se intensifica en la segunda quincena de octubre 2026",
    labels={"fecha_str": "Fecha", "cantidad": "N° noticias"}
)
fig2.show()

**Observación:** La cobertura de La República se intensifica hacia el 18 de octubre, coincidiendo con eventos políticos recientes como las elecciones regionales y la procesión del Señor de los Milagros.

In [79]:
# Gráfico 3 — Histograma de longitud de titulares
df["largo_titular"] = df["titular"].str.len()

fig3 = px.histogram(
    df, x="largo_titular", color="medio", nbins=30, barmode="overlay",
    title="El Comercio usa titulares más largos que La República en promedio",
    labels={"largo_titular": "N° caracteres", "medio": "Medio"}
)
fig3.show()

**Observación:** Ambos medios concentran sus titulares entre 50 y 150 caracteres. El Comercio tiene una cola más larga hacia la derecha, indicando que usa titulares más extensos y descriptivos, mientras La República tiende a ser más directa y concisa.

In [80]:
# Validación: 5 titulares más largos y más cortos
print("🔤 5 titulares MÁS LARGOS:")
print(df.nlargest(5, "largo_titular")[["titular","medio","etiqueta"]].to_string())

print("\n🔤 5 titulares MÁS CORTOS:")
print(df.nsmallest(5, "largo_titular")[["titular","medio","etiqueta"]].to_string())

🔤 5 titulares MÁS LARGOS:
                                                                                                                                                                                                                                                                                                                                                                titular         medio            etiqueta
260  Superávit comercial del Perú superó los US$30.000 millones entre enero y agosto, según BCRSuperávit comercial del Perú superó los US$30.000 millones entre enero y agosto, según BCR. El saldo fue mayor en US$12.291 millones al del mismo período de 2025. Solo en agosto, las exportaciones crecieron 36,9%, impulsadas principalmente por los mayores precios.   El Comercio            economia
303                                                                                                                                                                                  “El j

**Observación:** Los titulares más largos pertenecen a El Comercio y son muy descriptivos (+300 caracteres), coherente con el histograma. Los titulares más cortos como "Temblor en Perú" o "Partidos de hoy" son titulares genéricos que no encajan bien en ninguna categoría — estos serían candidatos a descarte en una limpieza más estricta.

In [81]:
# Exportación final
cols = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
df = df[cols]
df.to_csv("datos/noticias.csv", index=False)

print(f"✅ datos/noticias.csv exportado")
print(f"Shape: {df.shape}")
df.head()

✅ datos/noticias.csv exportado
Shape: (400, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Señor de los Milagros,Sigue EN VIVO la procesión del Señor de los Mi...,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/don...,2026-10-18,politica
1,Sporting Cristal vs Comerciantes Unidos,El equipo dirigido por Roberto Mosquera se hiz...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
2,Alianza Atlético vs Universitario,El enfrentamiento por la jornada 12 del Torneo...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
3,Vóley Alianza Lima vs Rebaza Acosta,El partido por la primera jornada de la Liga P...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
4,Temblor en Perú,Consulta dónde ocurrió el último temblor este ...,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/tem...,2026-10-18,politica


In [83]:
df.to_csv("datos/noticias.csv", index=False)
print(f"✅ CSV actualizado: {df.shape}")
df.head()

✅ CSV actualizado: (400, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Señor de los Milagros,Sigue EN VIVO la procesión del Señor de los Mi...,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/don...,2026-10-18,politica
1,Sporting Cristal vs Comerciantes Unidos,El equipo dirigido por Roberto Mosquera se hiz...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
2,Alianza Atlético vs Universitario,El enfrentamiento por la jornada 12 del Torneo...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
3,Vóley Alianza Lima vs Rebaza Acosta,El partido por la primera jornada de la Liga P...,La Republica,politica,https://larepublica.pe/deportes/2026/10/18/en-...,2026-10-18,politica
4,Temblor en Perú,Consulta dónde ocurrió el último temblor este ...,La Republica,politica,https://larepublica.pe/sociedad/2026/10/18/tem...,2026-10-18,politica
